# Week 10: Object Detection

**Lecture 16 — October 20:** Object Detection I  
**Lecture 17 — October 22:** Object Detection II

Classification answers *what is in the image?* Detection must also answer *where is each object?* This notebook builds that idea through the R-CNN family using **Pascal VOC 2007**, the classic 20-class detection benchmark used by the original R-CNN work.

## One historical motivation, then detectors

Szegedy, Toshev, and Erhan's 2013 paper [Deep Neural Networks for Object Detection](https://papers.nips.cc/paper_files/paper/2013/hash/f7cade80b7cc92b991cf4d2806d6bd78-Abstract.html) framed localization as regression from an image to an object mask. It helped establish that learned CNN features could replace hand-designed detection pipelines. We mention that idea briefly and now focus on the detector lineage that became especially influential:

1. **R-CNN:** propose regions, run a CNN on every crop, and classify each region.
2. **Fast R-CNN:** run the CNN once and pool every region from the shared feature map.
3. **Faster R-CNN:** replace the external proposal algorithm with a learned region proposal network.

## Setup

The dataset is downloaded to `../datasets` relative to the repository root. Run this notebook from its existing `Week-10-Object-Detection` directory, so the path is simply `../datasets`.

In [ ]:
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import torch
from IPython.display import display
from ipywidgets import FloatSlider, IntSlider, interact
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.datasets import VOCDetection
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.models.detection import (
    FasterRCNN_MobileNet_V3_Large_FPN_Weights,
    fasterrcnn_mobilenet_v3_large_fpn,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import box_iou, nms, roi_align
from torchvision.transforms.functional import normalize, pil_to_tensor, resize
from torchvision.utils import draw_bounding_boxes

torch.manual_seed(7)
random.seed(7)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIRECTORY = Path("../datasets")
CHECKPOINT_DIRECTORY = Path("checkpoints")
CHECKPOINT_DIRECTORY.mkdir(exist_ok=True)

print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")

## Pascal VOC 2007

VOC is far smaller than COCO but is not a toy or synthetic dataset. It contains varied natural images, multiple objects per image, and 20 familiar classes. `torchvision` downloads the official train/validation split and parses its XML annotations. We use a reproducible subset by default so the entire notebook is practical during class; increase `TRAIN_IMAGE_LIMIT` for a longer run.

In [ ]:
VOC_CLASSES = [
    "__background__", "aeroplane", "bicycle", "bird", "boat",
    "bottle", "bus", "car", "cat", "chair", "cow", "diningtable",
    "dog", "horse", "motorbike", "person", "pottedplant", "sheep",
    "sofa", "train", "tvmonitor",
]
CLASS_TO_INDEX = {name: index for index, name in enumerate(VOC_CLASSES)}

voc_train = VOCDetection(
    root=DATA_DIRECTORY, year="2007", image_set="train", download=True
)
voc_validation = VOCDetection(
    root=DATA_DIRECTORY, year="2007", image_set="val", download=True
)

def unpack_target(annotation):
    objects = annotation["annotation"].get("object", [])
    if isinstance(objects, dict):
        objects = [objects]

    boxes, labels = [], []
    for object_annotation in objects:
        box = object_annotation["bndbox"]
        boxes.append([float(box[key]) for key in ("xmin", "ymin", "xmax", "ymax")])
        labels.append(CLASS_TO_INDEX[object_annotation["name"]])
    return torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels)

print(f"Training images: {len(voc_train):,}")
print(f"Validation images: {len(voc_validation):,}")

In [ ]:
def show_voc_image(index=0):
    image, annotation = voc_train[index]
    boxes, labels = unpack_target(annotation)
    image_tensor = pil_to_tensor(image)
    label_names = [VOC_CLASSES[label] for label in labels]
    drawn = draw_bounding_boxes(image_tensor, boxes, labels=label_names, width=3)

    plt.figure(figsize=(10, 7))
    plt.imshow(drawn.permute(1, 2, 0))
    plt.axis("off")
    plt.show()

interact(
    show_voc_image,
    index=IntSlider(min=0, max=min(100, len(voc_train) - 1), step=1, value=0, continuous_update=False),
);

## Boxes, overlap, and duplicate removal

Intersection over union connects training and inference:

$$\operatorname{IoU}(A,B)=\frac{|A\cap B|}{|A\cup B|}.
$$

A candidate can be labeled positive when its IoU with a target is high. During inference, non-maximum suppression (NMS) keeps the highest-scoring box and removes lower-scoring duplicates whose IoU exceeds a threshold. Move the slider to see that a lower NMS threshold suppresses more boxes.

In [ ]:
demo_image, _ = voc_train[6]
demo_image = pil_to_tensor(demo_image)
demo_boxes = torch.tensor([[35, 40, 260, 300], [55, 55, 270, 310], [290, 80, 480, 330]], dtype=torch.float32)
demo_scores = torch.tensor([0.96, 0.78, 0.85])

def show_nms(iou_threshold=0.5):
    kept = nms(demo_boxes, demo_scores, iou_threshold)
    labels = [f"score={demo_scores[index]:.2f}" for index in kept]
    drawn = draw_bounding_boxes(demo_image, demo_boxes[kept], labels=labels, colors="yellow", width=4)
    print(f"Pairwise IoU:\n{box_iou(demo_boxes, demo_boxes).round(decimals=2)}")
    print(f"Kept proposal indices: {kept.tolist()}")
    plt.figure(figsize=(9, 6))
    plt.imshow(drawn.permute(1, 2, 0))
    plt.axis("off")
    plt.show()

interact(
    show_nms,
    iou_threshold=FloatSlider(min=0.1, max=0.9, step=0.05, value=0.5, continuous_update=False),
);

# R-CNN: train a classifier on proposed regions

Original R-CNN used an external region-proposal algorithm, warped every proposal to a fixed size, computed CNN features separately for every crop, and then classified each region. Here, jittered target boxes supply positive proposals and random boxes supply background proposals. This preserves the central learning problem while avoiding an extra legacy selective-search dependency.

The ImageNet-pretrained ResNet-18 backbone is frozen. We train a new 21-way region classifier—20 VOC classes plus background. Every access generates either a positive proposal near an object or a negative proposal with low IoU.

In [ ]:
def jitter_box(box, image_width, image_height):
    x_min, y_min, x_max, y_max = box.tolist()
    width, height = x_max - x_min, y_max - y_min
    noise = torch.randn(4) * torch.tensor([width, height, width, height]) * 0.08
    proposal = box + noise
    proposal[0].clamp_(0, image_width - 3)
    proposal[1].clamp_(0, image_height - 3)
    proposal[2].clamp_(proposal[0] + 2, image_width)
    proposal[3].clamp_(proposal[1] + 2, image_height)
    return proposal

def random_background_box(target_boxes, image_width, image_height):
    for _ in range(100):
        width = random.randint(max(20, image_width // 10), max(21, image_width // 2))
        height = random.randint(max(20, image_height // 10), max(21, image_height // 2))
        x_min = random.randint(0, max(0, image_width - width))
        y_min = random.randint(0, max(0, image_height - height))
        proposal = torch.tensor([x_min, y_min, x_min + width, y_min + height], dtype=torch.float32)
        if box_iou(proposal.unsqueeze(0), target_boxes).max() < 0.3:
            return proposal
    return torch.tensor([0, 0, image_width // 4, image_height // 4], dtype=torch.float32)

class VOCRegionDataset(Dataset):
    def __init__(self, voc_dataset, image_indices, samples_per_image=4):
        self.voc_dataset = voc_dataset
        self.image_indices = list(image_indices)
        self.samples_per_image = samples_per_image
        self.image_size = (224, 224)
        self.normalization = ResNet18_Weights.DEFAULT.transforms()

    def __len__(self):
        return len(self.image_indices) * self.samples_per_image

    def __getitem__(self, index):
        image, annotation = self.voc_dataset[self.image_indices[index // self.samples_per_image]]
        image_tensor = pil_to_tensor(image).float() / 255
        boxes, labels = unpack_target(annotation)
        image_height, image_width = image_tensor.shape[-2:]

        if index % 2 == 0:
            object_index = random.randrange(len(boxes))
            proposal = jitter_box(boxes[object_index], image_width, image_height)
            label = labels[object_index]
        else:
            proposal = random_background_box(boxes, image_width, image_height)
            label = torch.tensor(0)

        x_min, y_min, x_max, y_max = proposal.round().int().tolist()
        crop = image_tensor[:, y_min:y_max, x_min:x_max]
        crop = self.normalization(resize(crop, self.image_size, antialias=True))
        return crop, label

TRAIN_IMAGE_LIMIT = 1_000
VALIDATION_IMAGE_LIMIT = 250
region_train = VOCRegionDataset(voc_train, range(min(TRAIN_IMAGE_LIMIT, len(voc_train))))
region_validation = VOCRegionDataset(voc_validation, range(min(VALIDATION_IMAGE_LIMIT, len(voc_validation))))
region_train_loader = DataLoader(region_train, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
region_validation_loader = DataLoader(region_validation, batch_size=64, num_workers=4, pin_memory=True)

print(f"Region crops per epoch: {len(region_train):,} train / {len(region_validation):,} validation")

In [ ]:
rcnn = resnet18(weights=ResNet18_Weights.DEFAULT)
for parameter in rcnn.parameters():
    parameter.requires_grad = False
rcnn.fc = nn.Linear(rcnn.fc.in_features, len(VOC_CLASSES))
rcnn = rcnn.to(DEVICE)

loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(rcnn.fc.parameters(), lr=3e-4, weight_decay=1e-4)
RCNN_EPOCHS = 8
RCNN_PATIENCE = 2
best_validation_loss = float("inf")
best_state = None
epochs_without_improvement = 0

for epoch in range(1, RCNN_EPOCHS + 1):
    # Evaluation mode keeps the frozen backbone's batch-normalization statistics fixed.
    # Gradients still train the new classification layer.
    rcnn.eval()
    train_loss, train_correct, train_count = 0.0, 0, 0
    for crops, labels in region_train_loader:
        crops = crops.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = rcnn(crops)
        loss = loss_function(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(labels)
        train_correct += (logits.argmax(1) == labels).sum().item()
        train_count += len(labels)

    rcnn.eval()
    validation_loss, validation_correct, validation_count = 0.0, 0, 0
    with torch.inference_mode():
        for crops, labels in region_validation_loader:
            crops = crops.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            logits = rcnn(crops)
            validation_loss += loss_function(logits, labels).item() * len(labels)
            validation_correct += (logits.argmax(1) == labels).sum().item()
            validation_count += len(labels)

    train_loss /= train_count
    validation_loss /= validation_count
    print(
        f"Epoch {epoch:02d}/{RCNN_EPOCHS} | "
        f"loss {train_loss:.4f} | val loss {validation_loss:.4f} | "
        f"accuracy {train_correct / train_count:.1%} | "
        f"val accuracy {validation_correct / validation_count:.1%}"
    )

    if validation_loss < best_validation_loss - 1e-3:
        best_validation_loss = validation_loss
        best_state = deepcopy(rcnn.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= RCNN_PATIENCE:
            print(f"Early stopping after epoch {epoch}.")
            break

rcnn.load_state_dict(best_state)
torch.save(rcnn.state_dict(), CHECKPOINT_DIRECTORY / "week10_rcnn_voc2007.pt")

### R-CNN's bottleneck

The classifier genuinely learned from VOC regions, but each proposal requires a separate backbone pass. If an image has 2,000 proposals, R-CNN performs nearly the same convolution 2,000 times over heavily overlapping crops. The next cell times that repeated computation on a small set of proposals.

In [ ]:
sample_image, sample_annotation = voc_validation[3]
sample_tensor = pil_to_tensor(sample_image).float() / 255
sample_boxes, _ = unpack_target(sample_annotation)
height, width = sample_tensor.shape[-2:]
proposals = [jitter_box(sample_boxes[index % len(sample_boxes)], width, height) for index in range(32)]
proposal_crops = []
for proposal in proposals:
    x_min, y_min, x_max, y_max = proposal.round().int().tolist()
    crop = sample_tensor[:, y_min:y_max, x_min:x_max]
    proposal_crops.append(region_validation.normalization(resize(crop, [224, 224], antialias=True)))

with torch.inference_mode():
    rcnn_logits = rcnn(torch.stack(proposal_crops).to(DEVICE))
print(f"R-CNN ran the complete ResNet backbone {len(proposal_crops)} times—once per proposal.")

# Fast R-CNN: share the feature map

Fast R-CNN reverses the order: first compute one feature map for the full image, then use ROI pooling to obtain a fixed-size feature tensor for every proposal. `roi_align` is a smoother modern version of ROI pooling; it samples without rounding ROI boundaries to integer feature-map cells.

Fast R-CNN jointly predicts a class and class-specific box correction for each ROI. The expensive backbone is shared, but an external proposal method is still required.

In [ ]:
fast_backbone = nn.Sequential(*list(resnet18(weights=ResNet18_Weights.DEFAULT).children())[:-2]).to(DEVICE).eval()
normalized_image = normalize(
    sample_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
).unsqueeze(0).to(DEVICE)

with torch.inference_mode():
    feature_map = fast_backbone(normalized_image)  # One backbone pass for the entire image.
    spatial_scale = feature_map.shape[-1] / normalized_image.shape[-1]
    roi_features = roi_align(
        feature_map, [torch.stack(proposals).to(DEVICE)], output_size=(7, 7),
        spatial_scale=spatial_scale, aligned=True,
    )

print(f"Shared feature map: {tuple(feature_map.shape)}")
print(f"One pooled feature tensor per ROI: {tuple(roi_features.shape)}")
print("Fast R-CNN ran the backbone once, regardless of the number of proposals.")

# Faster R-CNN: learn the proposals

Faster R-CNN adds a region proposal network (RPN) over the shared feature map. At each anchor it predicts:

- **objectness:** whether an object is likely to be present; and
- **box offsets:** how the anchor should move and resize.

The highest-quality proposals flow into the Fast R-CNN head for final classification and box refinement. The total loss combines RPN objectness, RPN box regression, ROI classification, and ROI box regression. We start with torchvision's pretrained detector, replace its final predictor for VOC's 20 classes plus background, and fine-tune it.

In [ ]:
class VOCObjectDetectionDataset(Dataset):
    def __init__(self, voc_dataset, image_indices):
        self.voc_dataset = voc_dataset
        self.image_indices = list(image_indices)

    def __len__(self):
        return len(self.image_indices)

    def __getitem__(self, index):
        image, annotation = self.voc_dataset[self.image_indices[index]]
        image = pil_to_tensor(image).float() / 255
        boxes, labels = unpack_target(annotation)
        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        target = {
            "boxes": boxes, "labels": labels, "area": area,
            "iscrowd": torch.zeros(len(boxes), dtype=torch.int64),
            "image_id": torch.tensor(self.image_indices[index]),
        }
        return image, target

def detection_collate(batch):
    return tuple(zip(*batch))

DETECTION_TRAIN_LIMIT = 1_000
DETECTION_VALIDATION_LIMIT = 250
detection_train = VOCObjectDetectionDataset(voc_train, range(min(DETECTION_TRAIN_LIMIT, len(voc_train))))
detection_validation = VOCObjectDetectionDataset(voc_validation, range(min(DETECTION_VALIDATION_LIMIT, len(voc_validation))))
detection_train_loader = DataLoader(
    detection_train, batch_size=4, shuffle=True, num_workers=4, pin_memory=True, collate_fn=detection_collate
)
detection_validation_loader = DataLoader(
    detection_validation, batch_size=2, num_workers=4, pin_memory=True, collate_fn=detection_collate
)

In [ ]:
detector = fasterrcnn_mobilenet_v3_large_fpn(
    weights=FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT
)
predictor_input_features = detector.roi_heads.box_predictor.cls_score.in_features
detector.roi_heads.box_predictor = FastRCNNPredictor(predictor_input_features, len(VOC_CLASSES))
detector = detector.to(DEVICE)

optimizer = torch.optim.SGD(
    [parameter for parameter in detector.parameters() if parameter.requires_grad],
    lr=0.005, momentum=0.9, weight_decay=5e-4,
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
print(f"Trainable parameters: {sum(p.numel() for p in detector.parameters() if p.requires_grad):,}")

In [ ]:
DETECTION_EPOCHS = 6
DETECTION_PATIENCE = 2
best_validation_loss = float("inf")
best_detector_state = None
epochs_without_improvement = 0

for epoch in range(1, DETECTION_EPOCHS + 1):
    detector.train()
    running_loss = 0.0
    component_totals = {}

    for images, targets in detection_train_loader:
        images = [image.to(DEVICE, non_blocking=True) for image in images]
        targets = [{key: value.to(DEVICE) for key, value in target.items()} for target in targets]
        loss_dictionary = detector(images, targets)
        loss = sum(loss_dictionary.values())

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        for name, value in loss_dictionary.items():
            component_totals[name] = component_totals.get(name, 0.0) + value.item()

    epoch_loss = running_loss / len(detection_train_loader)

    # Detection models return their loss dictionary in training mode. With
    # gradients disabled, the same interface gives a useful validation loss.
    validation_loss = 0.0
    with torch.no_grad():
        for images, targets in detection_validation_loader:
            images = [image.to(DEVICE, non_blocking=True) for image in images]
            targets = [{key: value.to(DEVICE) for key, value in target.items()} for target in targets]
            validation_loss += sum(detector(images, targets).values()).item()
    validation_loss /= len(detection_validation_loader)
    scheduler.step()
    components = " | ".join(
        f"{name.replace('loss_', '')} {total / len(detection_train_loader):.3f}"
        for name, total in component_totals.items()
    )
    print(
        f"Epoch {epoch:02d}/{DETECTION_EPOCHS} | loss {epoch_loss:.4f} | "
        f"val loss {validation_loss:.4f} | "
        f"lr {optimizer.param_groups[0]['lr']:.2e} | {components}"
    )

    if validation_loss < best_validation_loss - 1e-3:
        best_validation_loss = validation_loss
        best_detector_state = deepcopy(detector.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= DETECTION_PATIENCE:
            print(f"Early stopping after epoch {epoch}.")
            break

detector.load_state_dict(best_detector_state)
torch.save(detector.state_dict(), CHECKPOINT_DIRECTORY / "week10_fasterrcnn_voc2007.pt")
print(f"Saved checkpoint with validation loss {best_validation_loss:.4f}.")

## Explore the trained detector inline

The score threshold controls which predictions are displayed. The NMS threshold is applied inside the detector, before its final predictions are returned. Lower it to suppress overlapping candidates more aggressively. No popup windows are used.

In [ ]:
def show_detection(index=0, score_threshold=0.5, nms_threshold=0.5):
    image, target = detection_validation[index]
    previous_nms_threshold = detector.roi_heads.nms_thresh
    detector.roi_heads.nms_thresh = nms_threshold
    detector.eval()
    with torch.inference_mode():
        prediction = detector([image.to(DEVICE)])[0]
    detector.roi_heads.nms_thresh = previous_nms_threshold

    keep = prediction["scores"] >= score_threshold
    boxes = prediction["boxes"][keep].cpu()
    scores = prediction["scores"][keep].cpu()
    labels = prediction["labels"][keep].cpu()
    captions = [f"{VOC_CLASSES[label]} {score:.2f}" for label, score in zip(labels, scores)]
    drawn = draw_bounding_boxes(
        (image * 255).byte(), boxes, labels=captions, colors="lime", width=3
    )

    plt.figure(figsize=(11, 8))
    plt.imshow(drawn.permute(1, 2, 0))
    plt.title(f"{len(boxes)} predictions")
    plt.axis("off")
    plt.show()

interact(
    show_detection,
    index=IntSlider(min=0, max=len(detection_validation) - 1, value=0, continuous_update=False),
    score_threshold=FloatSlider(min=0.05, max=0.95, step=0.05, value=0.5, continuous_update=False),
    nms_threshold=FloatSlider(min=0.1, max=0.9, step=0.05, value=0.5, continuous_update=False),
);

# Lecture 17: complete the progression, then contrast one-stage detectors

The first part consolidates R-CNN → Fast R-CNN → Faster R-CNN and inspects the trained model. The second part introduces one-stage detection:

- **Two-stage:** propose a smaller set of candidate regions, then classify and refine them.
- **One-stage:** predict boxes, objectness, and classes densely from feature maps.

A later extension will implement a YOLO-style head and target encoder. For now, the important comparison is computational: YOLO removes the separate proposal-and-ROI stage, gaining speed while taking on a harder dense matching and class-imbalance problem.

## Code takeaways

- Pascal VOC provides a manageable, established benchmark with real multi-object scenes and 20 classes.
- R-CNN turns detection into region classification, but repeated backbone computation makes it slow.
- Fast R-CNN shares one feature map and extracts fixed-size ROI features for every proposal.
- Faster R-CNN learns proposals with an RPN and trains proposal and ROI losses together.
- IoU is used for matching, NMS removes duplicates, and score thresholds trade precision against recall.
- The notebook selects CUDA automatically and moves images, targets, and both trainable models to the GPU.